# Loading

In [ ]:
from pathlib import Path
import re
import shutil
import zipfile

import gdown

DATASET_LINK = 'https://drive.google.com/file/d/1M8EPREjTLAqY3VAjfruDcxtaudXy4xm2/view?usp=share_link'

DATA_ROOT = Path.cwd()
ARCHIVE_DIR = DATA_ROOT / 'archives'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

FORCE_REDOWNLOAD = False
FORCE_REEXTRACT = False


def extract_drive_file_id(link: str) -> str:
    patterns = [r'/file/d/([a-zA-Z0-9_-]+)', r'id=([a-zA-Z0-9_-]+)']
    for pattern in patterns:
        match = re.search(pattern, link)
        if match:
            return match.group(1)
    raise ValueError(f'Could not parse Google Drive file id from: {link}')



def find_dataset_root(root: Path) -> Path | None:
    # The extracted archive can have variable nesting; find the actual dataset folder.
    for path in root.rglob('*'): # Look for any directory that contains JPEGImages or Annotations
        if path.is_dir() and (path / 'JPEGImages').exists() or (path / 'Annotations').exists():
            return path
    return None



def ensure_archive(link: str, output_zip: Path, force_download: bool = False) -> Path:
    if output_zip.exists() and not force_download:
        print(f'Archive already exists: {output_zip.name}')
        return output_zip
    if output_zip.exists() and force_download:
        output_zip.unlink()

    file_id = extract_drive_file_id(link)
    direct_url = f'https://drive.google.com/uc?id={file_id}'
    print(f'Downloading {output_zip.name}...')
    gdown.download(url=direct_url, output=str(output_zip), quiet=False)

    if not output_zip.exists() or output_zip.stat().st_size == 0:
        raise RuntimeError(f'Failed to download archive: {output_zip.name}')
    return output_zip



def install_voc_split(split_name: str, archive_path: Path, data_root: Path, force_extract: bool = False) -> Path:
    target = data_root / split_name
    if target.exists() and any(target.iterdir()) and not force_extract:
        print(f'Dataset already extracted: {target}')
        return target

    tmp_dir = data_root / '_tmp_extract' / split_name
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    print(f'Extracting {archive_path.name} to {tmp_dir}...')
    with zipfile.ZipFile(archive_path, 'r') as zf:
        zf.extractall(tmp_dir)

    dataset_dir = find_dataset_root(tmp_dir)
    if dataset_dir is None:
        # If expected dataset structure is not found, assume the zip directly contains the data
        # and move the contents of tmp_dir to target.
        if target.parent.exists() and force_extract:
            shutil.rmtree(target.parent)
        target.parent.mkdir(parents=True, exist_ok=True)

        if target.exists():
            shutil.rmtree(target)
        shutil.move(str(tmp_dir), str(target))
        print(f'Dataset extracted to: {target}')
        return target

    # If a dataset directory was found, move its content to the target.
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(dataset_dir, target)
    shutil.rmtree(tmp_dir)
    print(f'Dataset extracted to: {target}')
    return target


dataset_zip = ensure_archive(DATASET_LINK, ARCHIVE_DIR / 'dataset.zip', force_download=FORCE_REDOWNLOAD)
voc_dataset_root = install_voc_split('voc_dataset', dataset_zip, DATA_ROOT, force_extract=FORCE_REEXTRACT)

print('Dataset ready at:', voc_dataset_root)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_ROOT_CANDIDATES = [
    PROJECT_ROOT / 'voc_dataset' / 'version_2' / 'data',
    PROJECT_ROOT / 'version_2' / 'data',
]
for candidate in DATA_ROOT_CANDIDATES:
    if candidate.exists():
        DATA_ROOT = candidate
        break
else:
    DATA_ROOT = DATA_ROOT_CANDIDATES[0]

SPLIT_DIR = DATA_ROOT / 'clean_ids'
IMAGE_ROOT = DATA_ROOT / 'images'
ANNOTATION_ROOT = DATA_ROOT / 'annotations'
EXPECTED_SPLITS = ['train', 'val', 'dev_test', 'test']

missing_paths = [path for path in [SPLIT_DIR, IMAGE_ROOT, ANNOTATION_ROOT] if not path.exists()]
missing_split_dirs = [
    path
    for split_name in EXPECTED_SPLITS
    for path in [IMAGE_ROOT / split_name, ANNOTATION_ROOT / split_name]
    if not path.exists()
]

if missing_paths or missing_split_dirs:
    raise FileNotFoundError(
        'Expected version_2 split layout was not found. '
        'Check version_2/data/{clean_ids,images,annotations} and the per-split folders.'
    )

print('Dataset root:', DATA_ROOT)
print('Split files:', sorted(path.name for path in SPLIT_DIR.glob('*.txt')))
print('Image splits:', sorted(path.name for path in IMAGE_ROOT.iterdir() if path.is_dir()))
print('Annotation splits:', sorted(path.name for path in ANNOTATION_ROOT.iterdir() if path.is_dir()))


In [ ]:
# =============================================================================
#  CELL 1 — Pascal VOC version_2 EDA + produce cleaned manifests on disk
# =============================================================================
import os, json, hashlib, zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

PROJECT_ROOT = Path.cwd()
DATA_ROOT_CANDIDATES = [
    PROJECT_ROOT / 'voc_dataset' / 'version_2' / 'data',
    PROJECT_ROOT / 'version_2' / 'data',
]
for candidate in DATA_ROOT_CANDIDATES:
    if candidate.exists():
        DATA_ROOT = candidate
        break
else:
    DATA_ROOT = DATA_ROOT_CANDIDATES[0]
SPLIT_ID_DIR = DATA_ROOT / 'clean_ids'
IMAGE_ROOT = DATA_ROOT / 'images'
ANNOTATION_ROOT = DATA_ROOT / 'annotations'
MANIFEST_DIR = PROJECT_ROOT / 'manifests'
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

VOC_CLASSES = [
    'aeroplane','bicycle','bird','boat','bottle','bus','car','cat','chair',
    'cow','diningtable','dog','horse','motorbike','person','pottedplant',
    'sheep','sofa','train','tvmonitor'
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}
ID_TO_CLASS = {idx: name for name, idx in CLASS_TO_ID.items()}
PREDICTION_COLUMNS = [
    'image_id', 'split', 'source_model', 'class_id', 'cls',
    'xmin', 'ymin', 'xmax', 'ymax', 'score'
]

missing_paths = [path for path in [SPLIT_ID_DIR, IMAGE_ROOT, ANNOTATION_ROOT] if not path.exists()]
missing_split_dirs = [
    path
    for split_name in ['train', 'val', 'dev_test', 'test']
    for path in [IMAGE_ROOT / split_name, ANNOTATION_ROOT / split_name]
    if not path.exists()
]

if missing_paths or missing_split_dirs:
    raise FileNotFoundError(
        'Expected VOC layout was not found. '
        'Check version_2/data/{clean_ids,images,annotations} and split subfolders.'
    )


def read_ids_from_file(path: Path) -> list[str]:
    if not path.exists():
        return []

    # Handles both formats:
    # 1) "image_id"
    # 2) "image_id <object_index>" (VOC Layout lists)
    ids = []
    seen = set()
    for line in path.read_text().splitlines():
        tokens = line.strip().split()
        if not tokens:
            continue
        image_id = tokens[0]
        if image_id not in seen:
            ids.append(image_id)
            seen.add(image_id)
    return ids



def load_split_ids(split_name: str) -> list[str]:
    # Split IDs are materialized under version_2/data/clean_ids.
    split_path = SPLIT_ID_DIR / f'{split_name}.txt'
    split_ids = read_ids_from_file(split_path)
    if not split_ids:
        raise FileNotFoundError(f'Missing split file: {split_path}')
    return split_ids



def parse_annotation(xml_path: Path, image_id: str, split_name: str) -> tuple[dict, list[dict]]:
    root = ET.parse(xml_path).getroot()

    size_node = root.find('size')
    width = int(size_node.findtext('width', default='0')) if size_node is not None else 0
    height = int(size_node.findtext('height', default='0')) if size_node is not None else 0

    records = []
    for obj in root.findall('object'):
        cls = obj.findtext('name', default='').strip()
        if cls not in CLASS_TO_ID:
            continue

        box = obj.find('bndbox')
        if box is None:
            continue

        xmin = int(float(box.findtext('xmin', default='0')))
        ymin = int(float(box.findtext('ymin', default='0')))
        xmax = int(float(box.findtext('xmax', default='0')))
        ymax = int(float(box.findtext('ymax', default='0')))

        box_w = max(1, xmax - xmin + 1)
        box_h = max(1, ymax - ymin + 1)

        records.append({
            'image_id': image_id,
            'split': split_name,
            'cls': cls,
            'class_id': CLASS_TO_ID[cls],
            'xmin': xmin,
            'ymin': ymin,
            'xmax': xmax,
            'ymax': ymax,
            'box_w': box_w,
            'box_h': box_h,
            'box_area': box_w * box_h,
            'difficult': int(obj.findtext('difficult', default='0')),
            'truncated': int(obj.findtext('truncated', default='0')),
            'occluded': int(obj.findtext('occluded', default='0')),
            'width': width,
            'height': height,
        })

    image_meta = {
        'image_id': image_id,
        'split': split_name,
        'width': width,
        'height': height,
    }
    return image_meta, records



def build_split_tables(split_name: str, image_ids: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    image_rows, object_rows = [], []
    ann_dir = ANNOTATION_ROOT / split_name

    for image_id in image_ids:
        xml_path = ann_dir / f'{image_id}.xml'
        if not xml_path.exists():
            image_rows.append({'image_id': image_id, 'split': split_name, 'width': pd.NA, 'height': pd.NA})
            continue

        image_meta, rows = parse_annotation(xml_path, image_id, split_name)
        image_rows.append(image_meta)
        object_rows.extend(rows)

    images_df = pd.DataFrame(image_rows).drop_duplicates(subset=['image_id', 'split'])
    objects_df = pd.DataFrame(object_rows)

    if not objects_df.empty:
        objects_df['box_w'] = (objects_df['xmax'] - objects_df['xmin'] + 1).clip(lower=1)
        objects_df['box_h'] = (objects_df['ymax'] - objects_df['ymin'] + 1).clip(lower=1)
        objects_df['box_area'] = objects_df['box_w'] * objects_df['box_h']
        objects_df['aspect_ratio'] = objects_df['box_w'] / objects_df['box_h']
        image_area = (objects_df['width'] * objects_df['height']).replace(0, pd.NA)
        objects_df['norm_area'] = objects_df['box_area'] / image_area

    return images_df, objects_df


train_ids = load_split_ids('train')
val_ids = load_split_ids('val')
dev_test_ids = load_split_ids('dev_test')
test_ids = load_split_ids('test')

train_images, train_objects = build_split_tables('train', train_ids)
val_images, val_objects = build_split_tables('val', val_ids)
dev_test_images, dev_test_objects = build_split_tables('dev_test', dev_test_ids)
test_images, test_objects = build_split_tables('test', test_ids)

train_model_df = train_objects[(train_objects['xmax'] >= train_objects['xmin']) & (train_objects['ymax'] >= train_objects['ymin'])].copy()
val_model_df = val_objects[(val_objects['xmax'] >= val_objects['xmin']) & (val_objects['ymax'] >= val_objects['ymin'])].copy()
dev_test_model_df = dev_test_objects[(dev_test_objects['xmax'] >= dev_test_objects['xmin']) & (dev_test_objects['ymax'] >= dev_test_objects['ymin'])].copy()

train_model_df_ignoring_difficult = train_model_df[train_model_df['difficult'] == 0].reset_index(drop=True)
val_model_df_ignoring_difficult = val_model_df[val_model_df['difficult'] == 0].reset_index(drop=True)
dev_test_model_df_ignoring_difficult = dev_test_model_df[dev_test_model_df['difficult'] == 0].reset_index(drop=True)

print('Split counts:', len(train_ids), len(val_ids), len(dev_test_ids), len(test_ids))
print('train-val overlap:', len(set(train_ids) & set(val_ids)))
print('train-dev_test overlap:', len(set(train_ids) & set(dev_test_ids)))
print('val-dev_test overlap:', len(set(val_ids) & set(dev_test_ids)))
print('train-test overlap:', len(set(train_ids) & set(test_ids)))
print('val-test overlap:', len(set(val_ids) & set(test_ids)))
print('dev_test-test overlap:', len(set(dev_test_ids) & set(test_ids)))

duplicate_test_ids = []


# ---- 8. Save manifests to disk ----
(MANIFEST_DIR / 'train_ids.txt').write_text('\n'.join(train_ids))
(MANIFEST_DIR / 'val_ids.txt').write_text('\n'.join(val_ids))
(MANIFEST_DIR / 'dev_test_ids.txt').write_text('\n'.join(dev_test_ids))
(MANIFEST_DIR / 'test_ids.txt').write_text('\n'.join(test_ids))
(MANIFEST_DIR / 'duplicate_test_ids.txt').write_text('')

info = {
    'data_root':         str(DATA_ROOT),
    'image_root':        str(IMAGE_ROOT),
    'annotation_root':   str(ANNOTATION_ROOT),
    'classes':           VOC_CLASSES,
    'counts': {
        'train':               len(train_ids),
        'val':                 len(val_ids),
        'dev_test':            len(dev_test_ids),
        'test':                len(test_ids),
    },
    'manifest_files': {
        'train':      str(MANIFEST_DIR / 'train_ids.txt'),
        'val':        str(MANIFEST_DIR / 'val_ids.txt'),
        'dev_test':   str(MANIFEST_DIR / 'dev_test_ids.txt'),
        'test':       str(MANIFEST_DIR / 'test_ids.txt'),
    },
}
with open(MANIFEST_DIR / 'dataset_info.json', 'w') as f:
    json.dump(info, f, indent=2)

print(f'\n✅ Manifests saved to: {MANIFEST_DIR}')
print(json.dumps(info, indent=2))


# Training with NCA fine-tuning

In [ ]:
# =============================================================================
#  CELL 2 — Faster R-CNN + per-level NCA refiner
#  Trainable: NCA refiners + full box_predictor (cls_score + bbox_pred).
#  Frozen: backbone, FPN, RPN, RoI feature extractor.
# =============================================================================
import os, sys, subprocess, time, math, csv, json
import xml.etree.ElementTree as ET
from functools import partial
from pathlib import Path
from collections import OrderedDict
from PIL import Image

try:
    from torchmetrics.detection import MeanAveragePrecision
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torchmetrics', 'pycocotools'])
    from torchmetrics.detection import MeanAveragePrecision

from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
from torchvision.transforms import v2
from torchvision import tv_tensors
from torchvision.models.detection import (fasterrcnn_resnet50_fpn,
                                          FasterRCNN_ResNet50_FPN_Weights)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


# =============================================================================
#  Config
# =============================================================================
CONFIG = {
    "manifest_dir": str(Path.cwd() / 'manifests'),
    "image_size":   320,
    "batch_size":   8,
    "accum_steps":  4,
    "num_workers":  2,
    "num_classes":  20,

    "nca_steps":      16,
    "fire_rate":      0.5,
    "fire_rate_eval": 1.0,
    "attn_heads":     4,
    "attn_window":    3,
    "nca_hidden":     128,
    "nca_gamma_init": 0.1,

    "n_epochs":     24,
    "lr_nca":       5e-4,
    "lr_head":      1e-4,
    "weight_decay": 1e-4,
    "grad_clip":    1.0,
    "use_amp":      True,
    "amp_dtype":    "float16",
    "warmup_steps": 500,
    "val_every":    4,

    "score_threshold":   0.05,
    "nms_iou_threshold": 0.5,
    "max_detections":    100,
    "vis_num_images":    6,

    "best_map_min":    0.01,
    "resume_from":     None,
    "prev_best_map50": 0.0,

    "output_dir": "/content/NCA/outputs_frcnn_nca",
    "vis_dir":    "/content/NCA/outputs_frcnn_nca/vis",
    "model_path": "/content/NCA/models/frcnn_nca.pt",
    "run_name":   "FRCNN_PerLevelNCA_VOC",
}

with open(Path(CONFIG["manifest_dir"]) / "dataset_info.json") as f:
    DATASET_INFO = json.load(f)
required_keys = ["image_root", "annotation_root", "manifest_files"]
missing_keys = [k for k in required_keys if k not in DATASET_INFO]
if missing_keys:
    raise KeyError(
        "dataset_info.json is outdated for version_2-only loading. Re-run Cell 4 to regenerate manifests. Missing keys: "
        + ", ".join(missing_keys)
    )
print("Loaded manifests:")
print(json.dumps(DATASET_INFO["counts"], indent=2))

VOC_CLASSES  = DATASET_INFO["classes"]
CLASS_TO_IDX = {c: i for i, c in enumerate(VOC_CLASSES)}


# =============================================================================
#  Device + AMP
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}  (sm_{cap[0]}{cap[1]})")
else:
    cap = (0, 0)
    print("!! No GPU detected.\n")

_req = CONFIG.get("amp_dtype", "float16")
_AMP_DTYPE = torch.bfloat16 if (cap[0] >= 8 and _req == "bfloat16") else torch.float16

os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(CONFIG['vis_dir'],    exist_ok=True)
os.makedirs(os.path.dirname(CONFIG['model_path']), exist_ok=True)


# =============================================================================
#  Local MHA + NCA refiner
# =============================================================================
class LocalMHA(nn.Module):
    def __init__(self, channels, num_heads=4, window=3):
        super().__init__()
        assert channels % num_heads == 0
        self.C, self.heads = channels, num_heads
        self.d_head = channels // num_heads
        self.window = window; self.pad = window // 2
        self.n_ctx  = window * window
        self.W_q   = nn.Conv2d(channels, channels, 1)
        self.W_k   = nn.Conv2d(channels, channels, 1)
        self.W_v   = nn.Conv2d(channels, channels, 1)
        self.W_out = nn.Conv2d(channels, channels, 1)
        self.rel_bias = nn.Parameter(torch.zeros(num_heads, self.n_ctx))
        for m in [self.W_q, self.W_k, self.W_v, self.W_out]:
            nn.init.kaiming_uniform_(m.weight, a=math.sqrt(5))
            nn.init.zeros_(m.bias)

    def forward(self, x):
        B, C, H, W = x.shape; K = self.window; HW = H * W
        q = self.W_q(x)
        k_pad = F.pad(self.W_k(x), (self.pad,)*4, mode='reflect')
        v_pad = F.pad(self.W_v(x), (self.pad,)*4, mode='reflect')
        k_u = F.unfold(k_pad, kernel_size=K).view(B, self.heads, self.d_head, self.n_ctx, HW)
        v_u = F.unfold(v_pad, kernel_size=K).view(B, self.heads, self.d_head, self.n_ctx, HW)
        q   = q.view(B, self.heads, self.d_head, HW)
        scores = torch.einsum('bhdl,bhdnl->bhnl', q, k_u) / math.sqrt(self.d_head)
        scores = scores + self.rel_bias.view(1, self.heads, self.n_ctx, 1)
        attn = scores.softmax(dim=2)
        out = torch.einsum('bhnl,bhdnl->bhdl', attn, v_u).reshape(B, C, H, W)
        return self.W_out(out)


class NCARefiner(nn.Module):
    def __init__(self, channels, hidden, heads, window, fire_rate, gamma_init=0.1):
        super().__init__()
        self.channels  = channels
        self.fire_rate = fire_rate
        self.attn = LocalMHA(channels, num_heads=heads, window=window)
        self.fc0  = nn.Linear(channels, hidden)
        self.fc1  = nn.Linear(hidden, channels, bias=False)
        with torch.no_grad():
            self.fc1.weight.normal_(0.0, 0.01)
        self.norm  = nn.LayerNorm(channels)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))

    def _step(self, x_bchw, fire_rate):
        dx = self.attn(x_bchw)
        dx = dx.permute(0, 2, 3, 1).contiguous()
        dx = self.fc1(F.relu(self.fc0(dx)))
        if fire_rate < 1.0:
            mask = (torch.rand(dx.size(0), dx.size(1), dx.size(2), 1,
                               device=dx.device, dtype=dx.dtype) > (1.0 - fire_rate)
                   ).to(dx.dtype)
            dx = dx * mask
        dx = dx.permute(0, 3, 1, 2).contiguous()
        return x_bchw + dx

    def forward(self, x, steps, fire_rate=None):
        fr = self.fire_rate if fire_rate is None else fire_rate
        x0 = x; y = x
        for _ in range(steps):
            y = self._step(y, fr)
        delta = y - x0
        delta = delta.permute(0, 2, 3, 1).contiguous()
        delta = self.norm(delta)
        delta = delta.permute(0, 3, 1, 2).contiguous()
        return x0 + self.gamma * delta


class PerLevelNCARefiner(nn.Module):
    def __init__(self, level_keys, channels, hidden, heads, window,
                 fire_rate, gamma_init, steps):
        super().__init__()
        self.level_keys = list(level_keys)
        self.steps = steps
        self.refiners = nn.ModuleDict({
            k: NCARefiner(channels, hidden, heads, window, fire_rate, gamma_init)
            for k in self.level_keys
        })

    def forward(self, feats, fire_rate=None):
        out = OrderedDict()
        for k, v in feats.items():
            out[k] = self.refiners[k](v, self.steps, fire_rate=fire_rate) \
                     if k in self.refiners else v
        return out


class NCARefinedBackbone(nn.Module):
    def __init__(self, fpn_backbone, refiner):
        super().__init__()
        self.body = fpn_backbone
        self.refiner = refiner
        self.out_channels = fpn_backbone.out_channels

    def forward(self, x):
        feats = self.body(x)
        fr = None if self.training else CONFIG["fire_rate_eval"]
        return self.refiner(feats, fire_rate=fr)


# =============================================================================
#  Build model
# =============================================================================
def build_model(cfg):
    weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
    model = fasterrcnn_resnet50_fpn(
        weights=weights,
        min_size=cfg["image_size"],
        max_size=cfg["image_size"],
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features,
                                                       cfg["num_classes"] + 1)

    fpn_out_channels = model.backbone.out_channels
    level_keys = ['0', '1', '2', '3', 'pool']
    refiner = PerLevelNCARefiner(
        level_keys=level_keys,
        channels=fpn_out_channels,
        hidden=cfg["nca_hidden"],
        heads=cfg["attn_heads"],
        window=cfg["attn_window"],
        fire_rate=cfg["fire_rate"],
        gamma_init=cfg["nca_gamma_init"],
        steps=cfg["nca_steps"],
    )
    model.backbone = NCARefinedBackbone(model.backbone, refiner)
    return model


def apply_freeze_policy(model):
    # Freeze everything, then re-enable the two trainable blocks.
    # No .train() override: FRCNN's ResNet-FPN uses FrozenBatchNorm2d which is
    # permanently in eval mode, and there are no other BN/Dropout layers in the
    # frozen parts. Overriding .train() would push RPN/RoIHeads into eval and
    # silently break loss computation.
    for p in model.parameters():
        p.requires_grad = False
    for p in model.backbone.refiner.parameters():
        p.requires_grad = True
    for p in model.roi_heads.box_predictor.parameters():
        p.requires_grad = True
    return model


def param_groups(model, cfg):
    nca_params  = [p for p in model.backbone.refiner.parameters() if p.requires_grad]
    head_params = [p for p in model.roi_heads.box_predictor.parameters() if p.requires_grad]
    groups = [
        {"params": nca_params,  "lr": cfg["lr_nca"],
         "weight_decay": cfg["weight_decay"], "name": "nca"},
        {"params": head_params, "lr": cfg["lr_head"],
         "weight_decay": cfg["weight_decay"], "name": "box_predictor"},
    ]
    n_nca  = sum(p.numel() for p in nca_params)
    n_head = sum(p.numel() for p in head_params)
    n_all  = sum(p.numel() for p in model.parameters())
    print(f"Trainable: NCA={n_nca:,} | box_predictor={n_head:,} "
          f"| total={n_all:,} | fraction={(n_nca+n_head)/n_all:.4%}")
    return groups


# =============================================================================
#  Dataset
# =============================================================================
class VOCManifestDataset(data.Dataset):
    def __init__(self, image_root, annotation_root, split_name, manifest_file, image_size=320, augment=False):
        self.img_dir = Path(image_root) / split_name
        self.ann_dir = Path(annotation_root) / split_name
        self.ids = [l.strip() for l in Path(manifest_file).read_text().splitlines()
                    if l.strip()]
        print(f"[VOCManifest:{split_name}] {manifest_file} -> {len(self.ids)} images")

        self.image_size = image_size
        tfs = [v2.Resize((image_size, image_size))]
        if augment:
            tfs += [v2.RandomHorizontalFlip(p=0.5),
                    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)]
        tfs += [v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
                v2.SanitizeBoundingBoxes()]
        self.tf = v2.Compose(tfs)

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        image  = Image.open(self.img_dir / f"{img_id}.jpg").convert("RGB")
        W0, H0 = image.size
        root   = ET.parse(self.ann_dir / f"{img_id}.xml").getroot()

        boxes_list, labels_list = [], []
        for o in root.findall("object"):
            if int(o.findtext("difficult") or 0) == 1: continue
            name = o.findtext("name")
            if name not in CLASS_TO_IDX: continue
            bb = o.find("bndbox")
            boxes_list.append([float(bb.findtext("xmin")), float(bb.findtext("ymin")),
                               float(bb.findtext("xmax")), float(bb.findtext("ymax"))])
            labels_list.append(CLASS_TO_IDX[name] + 1)

        if not boxes_list:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,),   dtype=torch.long)
        else:
            boxes  = torch.tensor(boxes_list,  dtype=torch.float32)
            labels = torch.tensor(labels_list, dtype=torch.long)

        boxes_tv = tv_tensors.BoundingBoxes(boxes, format='XYXY', canvas_size=(H0, W0))
        out = self.tf({'image': image, 'boxes': boxes_tv, 'labels': labels})
        img_chw  = out['image']
        boxes_t  = torch.as_tensor(out['boxes'], dtype=torch.float32)
        labels_t = out['labels'].long()
        return img_chw, {"boxes": boxes_t, "labels": labels_t}


def frcnn_collate(batch):
    return [b[0] for b in batch], [b[1] for b in batch]


# =============================================================================
#  Visualization + helpers
# =============================================================================
_CMAP = plt.get_cmap('tab20')

def denormalize_chw(img_chw):
    img = img_chw.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    return (img * 255).astype(np.uint8)


def visualize_predictions(model, imgs_list, targets_list, cfg, tag, save_path):
    was_training = model.training
    model.eval()
    with torch.no_grad():
        amp = cfg["use_amp"] and device.type == 'cuda'
        with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=amp):
            dets = model([im.to(device) for im in imgs_list])

    n = len(imgs_list)
    cols = min(3, n); rows = int(math.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).reshape(rows, cols)

    score_thr_vis = 0.3
    for i in range(n):
        ax = axes[i // cols, i % cols]
        ax.imshow(denormalize_chw(imgs_list[i])); ax.set_xticks([]); ax.set_yticks([])
        gt = targets_list[i]
        for bb, lb in zip(gt['boxes'], gt['labels']):
            x1, y1, x2, y2 = bb.tolist()
            cls_idx = int(lb) - 1
            ax.add_patch(mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                fill=False, edgecolor='lime', linewidth=2, linestyle='--'))
            name = VOC_CLASSES[cls_idx] if 0 <= cls_idx < len(VOC_CLASSES) else "?"
            ax.text(x1, y1-2, f"GT:{name}", color='lime', fontsize=7, weight='bold',
                    bbox=dict(facecolor='black', alpha=0.5, pad=1))
        d = dets[i]
        keep = d['scores'] >= score_thr_vis
        for bb, sc, lb in zip(d['boxes'][keep].cpu(),
                              d['scores'][keep].cpu(),
                              d['labels'][keep].cpu()):
            x1, y1, x2, y2 = bb.tolist()
            cls_idx = int(lb) - 1
            if cls_idx < 0 or cls_idx >= len(VOC_CLASSES): continue
            color = _CMAP(cls_idx % 20)
            ax.add_patch(mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                fill=False, edgecolor=color, linewidth=1.8))
            ax.text(x1, y2+2, f"{VOC_CLASSES[cls_idx]}:{sc:.2f}", color='white',
                    fontsize=7, bbox=dict(facecolor=color, alpha=0.7, pad=1))
    for j in range(n, rows * cols): axes[j // cols, j % cols].axis('off')
    fig.suptitle(tag, fontsize=12); fig.tight_layout()
    fig.savefig(save_path, bbox_inches='tight', dpi=110); plt.close(fig)
    if was_training: model.train()


def _fmt_hms(s):
    s = int(max(0, s))
    h, s = divmod(s, 3600); m, s = divmod(s, 60)
    return f"{h:d}h{m:02d}m{s:02d}s" if h else f"{m:d}m{s:02d}s"


def _targets_to_device(targets, device):
    return [{k: v.to(device, non_blocking=True) for k, v in t.items()}
            for t in targets]


def _dets_for_metric(dets):
    return [{'boxes':  d['boxes'].detach().cpu(),
             'scores': d['scores'].detach().cpu(),
             'labels': (d['labels'].detach().cpu() - 1)} for d in dets]


def _gt_for_metric(targets):
    return [{'boxes':  t['boxes'].detach().cpu(),
             'labels': (t['labels'].detach().cpu() - 1)} for t in targets]


# =============================================================================
#  Train
# =============================================================================
def train(cfg):
    accum     = max(1, int(cfg.get("accum_steps", 1)))
    val_every = max(1, int(cfg.get("val_every", 1)))
    mdir      = Path(cfg["manifest_dir"])

    train_ds = VOCManifestDataset(DATASET_INFO["image_root"], DATASET_INFO["annotation_root"],
                                  "train", Path(DATASET_INFO["manifest_files"]["train"]),
                                  image_size=cfg["image_size"], augment=True)
    val_ds   = VOCManifestDataset(DATASET_INFO["image_root"], DATASET_INFO["annotation_root"],
                                  "val", Path(DATASET_INFO["manifest_files"]["val"]),
                                  image_size=cfg["image_size"], augment=False)

    train_loader = data.DataLoader(
        train_ds, batch_size=cfg["batch_size"], shuffle=True,
        num_workers=cfg["num_workers"], pin_memory=True, collate_fn=frcnn_collate,
        persistent_workers=cfg["num_workers"] > 0, drop_last=True)
    val_loader = data.DataLoader(
        val_ds, batch_size=cfg["batch_size"], shuffle=False,
        num_workers=cfg["num_workers"], pin_memory=True, collate_fn=frcnn_collate,
        persistent_workers=cfg["num_workers"] > 0)
    vis_loader = data.DataLoader(
        val_ds, batch_size=cfg["vis_num_images"], shuffle=True,
        num_workers=0, collate_fn=frcnn_collate)
    vis_imgs_list, vis_targets_list = next(iter(vis_loader))

    model = build_model(cfg).to(device)
    model = apply_freeze_policy(model)

    resume_path = cfg.get("resume_from")
    if resume_path and os.path.exists(resume_path):
        sd = torch.load(resume_path, map_location=device)
        missing, unexpected = model.load_state_dict(sd, strict=False)
        print(f"✅ RESUMED from {resume_path}"
              f" | missing: {len(missing)} | unexpected: {len(unexpected)}")
    elif resume_path:
        print(f"⚠️  resume_from not found ({resume_path}) — training from scratch.")
    else:
        print("⚠️  resume_from is None — training from scratch.")

    groups    = param_groups(model, cfg)
    optimizer = optim.AdamW(groups)
    microbatches_per_epoch = len(train_loader)
    opt_steps_per_epoch    = max(1, microbatches_per_epoch // accum)
    total_opt_steps        = cfg["n_epochs"] * opt_steps_per_epoch
    warmup = int(cfg.get("warmup_steps", 0))

    def lr_lambda(step):
        if warmup > 0 and step < warmup: return (step + 1) / float(warmup)
        progress = (step - warmup) / max(1, total_opt_steps - warmup)
        progress = min(1.0, max(0.0, progress))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    amp = cfg["use_amp"] and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=(amp and _AMP_DTYPE == torch.float16))

    history = {k: [] for k in ['train_total','train_cls','train_reg','train_obj','train_rpn',
                               'mAP50','mAP','val_epoch']}
    global_step = 0
    best_map50  = float(cfg.get("prev_best_map50", 0.0))
    train_epoch_times, val_epoch_times, run_start = [], [], time.time()

    print(f"\nRun: {cfg['run_name']} | epochs={cfg['n_epochs']} | "
          f"train={len(train_ds)} val={len(val_ds)} | "
          f"AMP={amp} ({_AMP_DTYPE})\n")

    for epoch in range(cfg["n_epochs"]):
        model.train()
        ep_start = time.time()
        run = {'total':0.0,'cls':0.0,'reg':0.0,'obj':0.0,'rpn':0.0}
        n_micro = 0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{cfg['n_epochs']}",
                    total=microbatches_per_epoch, leave=False,
                    dynamic_ncols=True, mininterval=30.0)

        for it, (imgs, tgts) in enumerate(pbar):
            imgs = [im.to(device, non_blocking=True) for im in imgs]
            tgts = _targets_to_device(tgts, device)
            with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=amp):
                loss_dict  = model(imgs, tgts)
                if not (isinstance(loss_dict, dict) and len(loss_dict) > 0):
                    # Safety net: if FRCNN returned detections (i.e. not in train
                    # mode) skip this microbatch instead of crashing.
                    continue
                loss       = sum(loss_dict.values())
                loss_to_bp = loss / accum
            if torch.is_tensor(loss) and torch.isfinite(loss):
                scaler.scale(loss_to_bp).backward()
                run['total'] += loss.item()
                run['cls']   += loss_dict.get('loss_classifier',  torch.tensor(0.0)).item()
                run['reg']   += loss_dict.get('loss_box_reg',     torch.tensor(0.0)).item()
                run['obj']   += loss_dict.get('loss_objectness',  torch.tensor(0.0)).item()
                run['rpn']   += loss_dict.get('loss_rpn_box_reg', torch.tensor(0.0)).item()
                n_micro += 1
            if (it + 1) % accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for g in optimizer.param_groups for p in g['params']],
                    cfg["grad_clip"])
                scaler.step(optimizer); scaler.update(); scheduler.step()
                optimizer.zero_grad(set_to_none=True); global_step += 1

            denom = max(1, n_micro)
            pbar.set_postfix({
                'L':   f"{run['total']/denom:.3f}",
                'cls': f"{run['cls']/denom:.3f}",
                'reg': f"{run['reg']/denom:.3f}",
                'obj': f"{run['obj']/denom:.3f}",
                'lr_nca':  f"{optimizer.param_groups[0]['lr']:.1e}",
            })
        pbar.close()

        train_epoch_times.append(time.time() - ep_start)
        denom = max(1, n_micro)
        history['train_total'].append(run['total']/denom)
        history['train_cls'].append(run['cls']/denom)
        history['train_reg'].append(run['reg']/denom)
        history['train_obj'].append(run['obj']/denom)
        history['train_rpn'].append(run['rpn']/denom)

        is_last = (epoch + 1) == cfg["n_epochs"]
        do_val  = ((epoch + 1) % val_every == 0) or is_last

        if do_val:
            val_start = time.time()
            model.eval()
            metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
            vbar = tqdm(val_loader, desc=f"  val Ep {epoch+1}",
                        total=len(val_loader), leave=False, dynamic_ncols=True)
            with torch.no_grad():
                for imgs, tgts in vbar:
                    imgs_d = [im.to(device, non_blocking=True) for im in imgs]
                    with torch.autocast(device_type=device.type,
                                        dtype=_AMP_DTYPE, enabled=amp):
                        dets = model(imgs_d)
                    metric.update(_dets_for_metric(dets), _gt_for_metric(tgts))
            vbar.close()
            m = metric.compute()
            map50, map_ = float(m['map_50']), float(m['map'])
            history['mAP50'].append(map50)
            history['mAP'].append(map_)
            history['val_epoch'].append(epoch + 1)
            val_epoch_times.append(time.time() - val_start)

            mean_train = sum(train_epoch_times) / len(train_epoch_times)
            mean_val   = sum(val_epoch_times)   / len(val_epoch_times)
            remaining_epochs     = cfg["n_epochs"] - epoch - 1
            remaining_val_epochs = sum(
                1 for e in range(epoch + 1, cfg["n_epochs"])
                if ((e + 1) % val_every == 0) or ((e + 1) == cfg["n_epochs"]))
            run_eta = remaining_epochs * mean_train + remaining_val_epochs * mean_val

            print(f"  Ep {epoch+1:2d} done in {_fmt_hms(time.time()-ep_start)}"
                  f" (train {_fmt_hms(train_epoch_times[-1])},"
                  f" val {_fmt_hms(val_epoch_times[-1])})"
                  f" | Train: {history['train_total'][-1]:.3f}"
                  f" | mAP@.5: {map50:.3f} | mAP: {map_:.3f}"
                  f" | elapsed {_fmt_hms(time.time()-run_start)}"
                  f" | ETA {_fmt_hms(run_eta)}")

            sp = os.path.join(cfg["vis_dir"], f"epoch_{epoch+1:03d}.png")
            visualize_predictions(model, vis_imgs_list, vis_targets_list, cfg,
                tag=f"Ep {epoch+1} | mAP@.5 {map50:.3f}", save_path=sp)

            torch.save(model.state_dict(), cfg["model_path"])
            if map50 > best_map50 and map50 >= cfg["best_map_min"]:
                best_map50 = map50
                best_path = cfg["model_path"].replace(".pt", "_best.pt")
                torch.save(model.state_dict(), best_path)
                print(f"  🌟 New best mAP@.5 ({map50:.4f})! Saved to {best_path}")
            else:
                print(f"  Checkpoint -> {cfg['model_path']} "
                      f"(best so far: {best_map50:.4f})")
        else:
            torch.save(model.state_dict(), cfg["model_path"])
            mean_train = sum(train_epoch_times) / len(train_epoch_times)
            mean_val   = (sum(val_epoch_times) / len(val_epoch_times)
                          if val_epoch_times else 0.0)
            remaining_epochs     = cfg["n_epochs"] - epoch - 1
            remaining_val_epochs = sum(
                1 for e in range(epoch + 1, cfg["n_epochs"])
                if ((e + 1) % val_every == 0) or ((e + 1) == cfg["n_epochs"]))
            run_eta = remaining_epochs * mean_train + remaining_val_epochs * mean_val
            print(f"  Ep {epoch+1:2d} done in {_fmt_hms(time.time()-ep_start)}"
                  f" | Train: {history['train_total'][-1]:.3f}"
                  f" | (no val this epoch)"
                  f" | elapsed {_fmt_hms(time.time()-run_start)}"
                  f" | ETA {_fmt_hms(run_eta)}")

    print(f"\nTotal run time: {_fmt_hms(time.time() - run_start)}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_total'], label='train total')
    axes[0].plot(history['train_cls'],   label='cls')
    axes[0].plot(history['train_reg'],   label='reg')
    axes[0].plot(history['train_obj'],   label='obj')
    axes[0].plot(history['train_rpn'],   label='rpn_reg')
    axes[0].set_title('Training losses'); axes[0].legend()
    if history['val_epoch']:
        val_x = [e - 1 for e in history['val_epoch']]
        axes[1].plot(val_x, history['mAP50'], 'o-', label='mAP@.5')
        axes[1].plot(val_x, history['mAP'],   'o-', label='mAP@[.5:.95]')
        axes[1].set_title('Validation mAP'); axes[1].legend()
    fig.savefig(os.path.join(cfg["output_dir"], f"{cfg['run_name']}_curves.png"),
                bbox_inches='tight')
    plt.show()

    with open(os.path.join(cfg["output_dir"], f"{cfg['run_name']}_history.csv"),
              'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['epoch','train_total','train_cls','train_reg','train_obj','train_rpn',
                    'mAP50','mAP'])
        val_lookup = {e: i for i, e in enumerate(history['val_epoch'])}
        for i in range(cfg["n_epochs"]):
            ep = i + 1
            row = [ep, history['train_total'][i], history['train_cls'][i],
                   history['train_reg'][i], history['train_obj'][i],
                   history['train_rpn'][i]]
            if ep in val_lookup:
                vi = val_lookup[ep]
                row += [history['mAP50'][vi], history['mAP'][vi]]
            else:
                row += ['','']
            w.writerow(row)

    return model, history


# =============================================================================
#  Test eval
# =============================================================================
def evaluate_test(cfg, checkpoint_path=None):
    test_ds = VOCManifestDataset(
        DATASET_INFO["image_root"], DATASET_INFO["annotation_root"],
        "test", Path(DATASET_INFO["manifest_files"]["test"]),
        image_size=cfg["image_size"], augment=False)
    test_loader = data.DataLoader(
        test_ds, batch_size=cfg["batch_size"], shuffle=False,
        num_workers=cfg["num_workers"], pin_memory=True, collate_fn=frcnn_collate)

    model = build_model(cfg).to(device)
    model = apply_freeze_policy(model)
    ckpt = checkpoint_path or cfg["model_path"].replace(".pt", "_best.pt")
    if not os.path.exists(ckpt): ckpt = cfg["model_path"]
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=device), strict=False)
        print(f"\n=== TEST EVAL === {ckpt} on {len(test_ds)} clean test images")
    else:
        print(f"\n=== TEST EVAL === (no checkpoint found; evaluating fresh weights)")
    model.eval()

    amp = cfg["use_amp"] and device.type == 'cuda'
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox',
                                  class_metrics=True)
    with torch.no_grad():
        for imgs, tgts in tqdm(test_loader, desc="test"):
            imgs_d = [im.to(device, non_blocking=True) for im in imgs]
            with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=amp):
                dets = model(imgs_d)
            metric.update(_dets_for_metric(dets), _gt_for_metric(tgts))
    m = metric.compute()
    print(f"\nTEST mAP@.5     : {float(m['map_50']):.4f}")
    print(f"TEST mAP@[.5:.95]: {float(m['map']):.4f}")
    if 'map_per_class' in m and m['map_per_class'].numel() == len(VOC_CLASSES):
        print("\nPer-class mAP@[.5:.95]:")
        for cname, v in zip(VOC_CLASSES, m['map_per_class'].tolist()):
            print(f"  {cname:14s}: {v:.3f}")

    with open(os.path.join(cfg["output_dir"],
                           f"{cfg['run_name']}_test_results.json"), 'w') as f:
        json.dump({'mAP_50': float(m['map_50']), 'mAP': float(m['map']),
                   'checkpoint': ckpt, 'n_images': len(test_ds)}, f, indent=2)
    return m


# =============================================================================
#  Run
# =============================================================================
model, history = train(CONFIG)
if history['mAP50']:
    print(f"\nBest val mAP@.5     : {max(history['mAP50']):.4f}")
    print(f"Best val mAP@.5:.95 : {max(history['mAP']):.4f}")

evaluate_test(CONFIG)

Ablation study below

In [ ]:
!pip install torchmetrics
import os
import sys
import math
import json
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import OrderedDict
import pandas as pd
from PIL import Image

# Suppress some noisy warnings if desired
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torchvision.transforms import v2
from torchvision import tv_tensors
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection import MeanAveragePrecision
from tqdm.auto import tqdm

# =============================================================================
#  1. Config & Setup
# =============================================================================
CONFIG = {
    "manifest_dir": str(Path.cwd() / 'manifests'),
    "image_size":   320,
    "batch_size":   8,
    "num_workers":  2,
    "num_classes":  20,

    "nca_steps":      16,
    "fire_rate":      0.5,
    "fire_rate_eval": 1.0,
    "attn_heads":     4,
    "attn_window":    3,
    "nca_hidden":     128,
    "nca_gamma_init": 0.1,

    "use_amp":      True,
    "amp_dtype":    "float16",

    "model_path": "/content/NCA/models/frcnn_nca.pt",
    "run_name":   "FRCNN_PerLevelNCA_VOC",
}

# Load Dataset Info
dataset_info_path = Path(CONFIG["manifest_dir"]) / "dataset_info.json"
if not dataset_info_path.exists():
    raise FileNotFoundError(f"Cannot find dataset_info.json at {dataset_info_path}. Please check paths.")

with open(dataset_info_path) as f:
    DATASET_INFO = json.load(f)

VOC_CLASSES  = DATASET_INFO["classes"]
CLASS_TO_IDX = {c: i for i, c in enumerate(VOC_CLASSES)}

# Device & AMP setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
_req = CONFIG.get("amp_dtype", "float16")
_AMP_DTYPE = torch.bfloat16 if (cap[0] >= 8 and _req == "bfloat16") else torch.float16

# =============================================================================
#  2. Model Architecture
# =============================================================================
class LocalMHA(nn.Module):
    def __init__(self, channels, num_heads=4, window=3):
        super().__init__()
        self.C, self.heads = channels, num_heads
        self.d_head = channels // num_heads
        self.window = window; self.pad = window // 2
        self.n_ctx  = window * window
        self.W_q   = nn.Conv2d(channels, channels, 1)
        self.W_k   = nn.Conv2d(channels, channels, 1)
        self.W_v   = nn.Conv2d(channels, channels, 1)
        self.W_out = nn.Conv2d(channels, channels, 1)
        self.rel_bias = nn.Parameter(torch.zeros(num_heads, self.n_ctx))

    def forward(self, x):
        B, C, H, W = x.shape; K = self.window; HW = H * W
        q = self.W_q(x)
        k_pad = F.pad(self.W_k(x), (self.pad,)*4, mode='reflect')
        v_pad = F.pad(self.W_v(x), (self.pad,)*4, mode='reflect')
        k_u = F.unfold(k_pad, kernel_size=K).view(B, self.heads, self.d_head, self.n_ctx, HW)
        v_u = F.unfold(v_pad, kernel_size=K).view(B, self.heads, self.d_head, self.n_ctx, HW)
        q   = q.view(B, self.heads, self.d_head, HW)
        scores = torch.einsum('bhdl,bhdnl->bhnl', q, k_u) / math.sqrt(self.d_head)
        scores = scores + self.rel_bias.view(1, self.heads, self.n_ctx, 1)
        attn = scores.softmax(dim=2)
        out = torch.einsum('bhnl,bhdnl->bhdl', attn, v_u).reshape(B, C, H, W)
        return self.W_out(out)

class NCARefiner(nn.Module):
    def __init__(self, channels, hidden, heads, window, fire_rate, gamma_init=0.1):
        super().__init__()
        self.channels  = channels
        self.fire_rate = fire_rate
        self.attn = LocalMHA(channels, num_heads=heads, window=window)
        self.fc0  = nn.Linear(channels, hidden)
        self.fc1  = nn.Linear(hidden, channels, bias=False)
        self.norm  = nn.LayerNorm(channels)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))

    def _step(self, x_bchw, fire_rate):
        dx = self.attn(x_bchw)
        dx = dx.permute(0, 2, 3, 1).contiguous()
        dx = self.fc1(F.relu(self.fc0(dx)))
        if fire_rate < 1.0:
            mask = (torch.rand(dx.size(0), dx.size(1), dx.size(2), 1,
                               device=dx.device, dtype=dx.dtype) > (1.0 - fire_rate)).to(dx.dtype)
            dx = dx * mask
        dx = dx.permute(0, 3, 1, 2).contiguous()
        return x_bchw + dx

    def forward(self, x, steps, fire_rate=None):
        fr = self.fire_rate if fire_rate is None else fire_rate
        x0 = x; y = x
        for _ in range(steps):
            y = self._step(y, fr)
        delta = y - x0
        delta = delta.permute(0, 2, 3, 1).contiguous()
        delta = self.norm(delta)
        delta = delta.permute(0, 3, 1, 2).contiguous()
        return x0 + self.gamma * delta

class PerLevelNCARefiner(nn.Module):
    def __init__(self, level_keys, channels, hidden, heads, window, fire_rate, gamma_init, steps):
        super().__init__()
        self.level_keys = list(level_keys)
        self.steps = steps
        self.refiners = nn.ModuleDict({
            k: NCARefiner(channels, hidden, heads, window, fire_rate, gamma_init)
            for k in self.level_keys
        })

    def forward(self, feats, fire_rate=None):
        out = OrderedDict()
        for k, v in feats.items():
            out[k] = self.refiners[k](v, self.steps, fire_rate=fire_rate) if k in self.refiners else v
        return out

class NCARefinedBackbone(nn.Module):
    def __init__(self, fpn_backbone, refiner):
        super().__init__()
        self.body = fpn_backbone
        self.refiner = refiner
        self.out_channels = fpn_backbone.out_channels

    def forward(self, x):
        feats = self.body(x)
        fr = None if self.training else CONFIG["fire_rate_eval"]
        return self.refiner(feats, fire_rate=fr)

def build_model(cfg):
    weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
    model = fasterrcnn_resnet50_fpn(weights=weights, min_size=cfg["image_size"], max_size=cfg["image_size"])
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, cfg["num_classes"] + 1)

    level_keys = ['0', '1', '2', '3', 'pool']
    refiner = PerLevelNCARefiner(
        level_keys=level_keys, channels=model.backbone.out_channels,
        hidden=cfg["nca_hidden"], heads=cfg["attn_heads"], window=cfg["attn_window"],
        fire_rate=cfg["fire_rate"], gamma_init=cfg["nca_gamma_init"], steps=cfg["nca_steps"]
    )
    model.backbone = NCARefinedBackbone(model.backbone, refiner)
    return model

# =============================================================================
#  3. Dataset & Helpers
# =============================================================================
class VOCManifestDataset(data.Dataset):
    def __init__(self, image_root, annotation_root, split_name, manifest_file, image_size=320, augment=False):
        self.img_dir = Path(image_root) / split_name
        self.ann_dir = Path(annotation_root) / split_name
        self.ids = [l.strip() for l in Path(manifest_file).read_text().splitlines() if l.strip()]

        tfs = [v2.Resize((image_size, image_size)), v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.SanitizeBoundingBoxes()]
        self.tf = v2.Compose(tfs)

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        image  = Image.open(self.img_dir / f"{img_id}.jpg").convert("RGB")
        W0, H0 = image.size
        root   = ET.parse(self.ann_dir / f"{img_id}.xml").getroot()

        boxes_list, labels_list = [], []
        for o in root.findall("object"):
            if int(o.findtext("difficult") or 0) == 1: continue
            name = o.findtext("name")
            if name not in CLASS_TO_IDX: continue
            bb = o.find("bndbox")
            boxes_list.append([float(bb.findtext("xmin")), float(bb.findtext("ymin")),
                               float(bb.findtext("xmax")), float(bb.findtext("ymax"))])
            labels_list.append(CLASS_TO_IDX[name] + 1)

        boxes  = torch.tensor(boxes_list,  dtype=torch.float32) if boxes_list else torch.zeros((0, 4), dtype=torch.float32)
        labels = torch.tensor(labels_list, dtype=torch.long) if labels_list else torch.zeros((0,),   dtype=torch.long)

        boxes_tv = tv_tensors.BoundingBoxes(boxes, format='XYXY', canvas_size=(H0, W0))
        out = self.tf({'image': image, 'boxes': boxes_tv, 'labels': labels})
        return out['image'], {"boxes": torch.as_tensor(out['boxes'], dtype=torch.float32), "labels": out['labels'].long()}

def frcnn_collate(batch):
    return [b[0] for b in batch], [b[1] for b in batch]

def _dets_for_metric(dets):
    return [{'boxes': d['boxes'].detach().cpu(), 'scores': d['scores'].detach().cpu(), 'labels': (d['labels'].detach().cpu() - 1)} for d in dets]

def _gt_for_metric(targets):
    return [{'boxes': t['boxes'].detach().cpu(), 'labels': (t['labels'].detach().cpu() - 1)} for t in targets]

# =============================================================================
#  4. Ablation Runner
# =============================================================================
def run_nca_ablations(cfg):
    print("\n" + "="*50)
    print(" 🕵️‍♂️ RUNNING NCA ABLATION TESTS")
    print("="*50)

    # Load data
    mdir = Path(cfg["manifest_dir"])
    val_ds = VOCManifestDataset(DATASET_INFO["image_root"], DATASET_INFO["annotation_root"], "val", Path(DATASET_INFO["manifest_files"]["val"]), image_size=cfg["image_size"], augment=False)
    val_loader = data.DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=cfg["num_workers"], pin_memory=True, collate_fn=frcnn_collate)

    # Load Model
    model = build_model(cfg).to(device)
    ckpt = cfg["model_path"].replace(".pt", "_best.pt")
    if not os.path.exists(ckpt):
        ckpt = cfg["model_path"]

    if os.path.exists(ckpt):
        print(f"Loading weights from: {ckpt}")
        model.load_state_dict(torch.load(ckpt, map_location=device), strict=False)
    else:
        print(f"⚠️ Warning: No checkpoint found at {ckpt}. Running with randomly initialized weights!")

    model.eval()

    # --- TEST 1: Learned Gamma Values ---
    print("\n--- TEST 1: Learned Gamma Values ---")
    print("If gamma is near 0.0, the model learned to completely bypass the NCA.")
    for level, refiner in model.backbone.refiner.refiners.items():
        g_val = refiner.gamma.item()
        print(f"  Level {level} Gamma: {g_val:.4f} " + ("⚠️ (Near Zero!)" if abs(g_val) < 0.005 else "✅ (Active)"))

    def eval_loop(desc):
        metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
        amp = cfg["use_amp"] and device.type == 'cuda'
        with torch.no_grad():
            for imgs, tgts in tqdm(val_loader, desc=desc, leave=False):
                imgs_d = [im.to(device, non_blocking=True) for im in imgs]
                with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=amp):
                    dets = model(imgs_d)
                metric.update(_dets_for_metric(dets), _gt_for_metric(tgts))
        return metric.compute()

    results_table = []

    # --- TEST 2: Step Count Ablation ---
    print("\n--- TEST 2: Step Count Ablation ---")
    orig_fire_rate = cfg.get("fire_rate_eval", 1.0)
    cfg["fire_rate_eval"] = 1.0 # Force deterministic eval for step test

    test_steps = [0, 1, 4, 8, 16, 32]
    for steps in test_steps:
        model.backbone.refiner.steps = steps
        m = eval_loop(f"Steps={steps:02d}")
        map50, map_ = float(m['map_50']), float(m['map'])
        print(f"  Steps: {steps:02d} | mAP@.5: {map50:.4f} | mAP@[.5:.95]: {map_:.4f}")
        results_table.append({'Test': 'Steps', 'Value': steps, 'mAP@.5': map50, 'mAP': map_})

    # --- TEST 3: Fire Rate Ablation ---
    print("\n--- TEST 3: Fire Rate Ablation ---")
    model.backbone.refiner.steps = 16
    test_fire_rates = [0.0, 0.25, 0.5, 0.75, 1.0]

    for fr in test_fire_rates:
        cfg["fire_rate_eval"] = fr
        m = eval_loop(f"FireRate={fr:.2f}")
        map50, map_ = float(m['map_50']), float(m['map'])
        print(f"  Fire Rate: {fr:.2f} | mAP@.5: {map50:.4f} | mAP@[.5:.95]: {map_:.4f}")
        results_table.append({'Test': 'FireRate', 'Value': fr, 'mAP@.5': map50, 'mAP': map_})

    # Summary
    print("\n--- ABLATION SUMMARY ---")
    df = pd.DataFrame(results_table)
    print(df.to_string(index=False))

    return df

if __name__ == "__main__":
    run_nca_ablations(CONFIG)

steps and NCA correlation, check if NCA is important